# Lab 4 · Forest carbon

**Day 2 · about 25 minutes · Student notebook**

> **Goal.** Estimate the carbon stored in your province's forest — and learn why two published global datasets give answers that differ by 70%.

---

### How to work in this notebook

Each step gives you a **prompt card**. Copy it into Claude (or Gemini in Colab),
paste the code you get back into the empty cell below it, and run it.
Then read the **check** underneath and make sure your numbers are believable.

If the AI's code throws an error: copy the **whole** error message, paste it back to the
assistant with the sentence *"this is the error I got, please fix the code"*, and try again.
Do not retype the code by hand.


## The chain from tree to tonne of CO₂

    biomass (Mg/ha)  --x 0.47-->  carbon (Mg C/ha)  --x 44/12-->  CO2 equivalent (t CO2e)

- **0.47** — roughly 47% of dry wood is carbon. This is the IPCC default.
- **44/12 = 3.667** — one tonne of carbon, fully oxidised, becomes 3.667 tonnes of CO₂.

Every carbon number you will ever read walks this chain. The mistakes happen at the first arrow,
because **some datasets have already applied it and some have not.**

### Setup

In [ ]:
# Run this first, every session. Colab forgets everything when it recycles.
!pip install -q geemap

import ee, geemap

ee.Authenticate()                      # opens a link - sign in, paste the code back
ee.Initialize(project='YOUR-PROJECT-ID')   # <-- put YOUR project ID here

print('Earth Engine is ready.')

In [ ]:
# Study area: one Thai province. No shapefile upload needed.
PROVINCE = 'Nan'        # <-- change to your own province

aoi_fc = (ee.FeatureCollection('FAO/GAUL/2015/level1')
          .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))
          .filter(ee.Filter.eq('ADM1_NAME', PROVINCE)))
aoi = aoi_fc.geometry()

area_ha = aoi.area(maxError=100).divide(1e4).getInfo()
print(f'{PROVINCE}: {area_ha:,.0f} hectares')

### Step 1 — the trap

Read these two catalog entries carefully:

| Dataset | Band | Units |
|---|---|---|
| `NASA/ORNL/biomass_carbon_density/v1` | `agb` | Mg **C**/ha — *already carbon* |
| `LARSE/GEDI/GEDI04_B_002` | `MU` | Mg/ha — *biomass, not carbon* |

They are both "aboveground biomass" datasets. One has already done the ×0.47 and one has not.
Multiply the wrong one and your answer is off by a factor of two.

There is a second trap in the first dataset. Try loading it the obvious way:

In [ ]:
# This is what almost every AI assistant will write - and it fails:
try:
    ornl = ee.Image('NASA/ORNL/biomass_carbon_density/v1')
    print(ornl.bandNames().getInfo())
except Exception as e:
    print('ERROR:', e)

`Asset 'NASA/ORNL/biomass_carbon_density/v1' is not an Image.`

It is an **ImageCollection that happens to contain exactly one image** (the year 2010).
The catalog page does not shout about this. Here is the correct load:

In [ ]:
ornl = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').first()
print('bands:', ornl.bandNames().getInfo())

### Step 2 — mean carbon density

In [ ]:
forest = ee.Image('ESA/WorldCover/v200/2021').select('Map').eq(10)

o = (ornl.select(['agb', 'bgb']).updateMask(forest)
     .reduceRegion(ee.Reducer.mean(), aoi, scale=300,
                   maxPixels=1e9, bestEffort=True).getInfo())

print('ORNL, over forest only')
print(f"  aboveground : {o['agb']:.1f} Mg C/ha   <- already carbon, do NOT multiply by 0.47")
print(f"  belowground : {o['bgb']:.1f} Mg C/ha")
print(f"  total       : {o['agb'] + o['bgb']:.1f} Mg C/ha")

> ### ✓ Check your answer
>
> For Nan forest: aboveground **46.2**, belowground **13.4**, total **59.5 Mg C/ha**. Tropical forest carbon is typically 40–200 Mg C/ha, so this is plausible for a mixed deciduous landscape.

### Step 3 — total stock and CO₂ equivalent

#### 🤖 Prompt card — Total carbon stock in the province

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Calculate the TOTAL carbon stored in the forest of my province, and its CO2 equivalent.
RESOURCE  Use the image in `ornl` (bands agb and bgb, units Mg C/ha - ALREADY CARBON).
OUTLINE   Restrict to forest pixels using the mask in `forest`. Region is `aoi`.
MUST      Multiply carbon density by pixel area in hectares BEFORE summing,
          because pixel area varies with latitude. Use scale=300.
          Then convert carbon to CO2e by multiplying by 44/12.
          Do NOT apply a 0.47 carbon fraction - this dataset is already carbon.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print total tonnes of carbon and total tonnes of CO2e with thousands separators.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> Nan gives about **63.9 million t C** and **234 million t CO₂e**. If you got a number in the thousands, you probably forgot to multiply by pixel area. If you got trillions, you multiplied by area in m² instead of hectares.

### Step 4 — get a second opinion, and be unsettled by it

#### 🤖 Prompt card — Compare against a second carbon dataset

Copy everything in the box into your AI assistant, then paste its answer into the empty cell below.

```text
PURPOSE   Check my carbon figure against an independent dataset built from lidar.
RESOURCE  Use LARSE/GEDI/GEDI04_B_002, band MU.
          IMPORTANT: MU is in Mg/ha of BIOMASS, not carbon. ORNL agb was already carbon.
OUTLINE   Area is `aoi`, restricted to forest with the mask in `forest`. Use scale=1000.
MUST      Report GEDI mean biomass, then convert it to carbon with the 0.47 fraction,
          then print the ratio between that and the ORNL value in `o['agb']`.
          Also report what fraction of my province GEDI actually covers.
PLATFORM  Google Colab, Earth Engine Python API.
TEST      Print all four numbers with their units labelled.
```

In [ ]:
# Paste the code your AI assistant gave you here, then run it.


> ### ✓ Check your answer
>
> ### Two published datasets, same forest, 1.7× apart
> 
> Nan: ORNL says **46 Mg C/ha**. GEDI says **163 Mg/ha biomass → 77 Mg C/ha**. They differ by 67%.
> 
> This is not a bug in your code. Both are real, peer-reviewed, widely cited products. They differ
> because:
> 
> - **Different years.** ORNL is a year-2010 snapshot. GEDI flew 2019–2021. Nine years of growth
>   and change sit between them.
> - **Different instruments.** GEDI is a spaceborne **lidar** that measures canopy height directly.
>   ORNL is a harmonised blend of earlier maps.
> - **Different coverage.** GEDI only reports where its laser actually landed — about **60%** of
>   this province. ORNL covers everything.
> 
> **What a forester should do about it.** Never report a single carbon figure as if it were
> measured. Report a **range** with the source and year attached:
> 
> > *"Aboveground carbon in Nan forest is estimated at 46–77 Mg C/ha
> > (ORNL 2010; GEDI L4B 2019–2021)."*
> 
> That sentence is honest and defensible. A single number with three decimal places is neither.